In [ ]:
import pandas as pd
import requests
from io import BytesIO
import numpy as np
import os
import re

In [ ]:
realtor = pd.read_csv("https://econdata.s3-us-west-2.amazonaws.com/Reports/Core/RDC_Inventory_Core_Metrics_County_History.csv")
realtor_hotness = pd.read_csv("https://econdata.s3-us-west-2.amazonaws.com/Reports/Hotness/RDC_Inventory_Hotness_Metrics_County_History.csv")

url = "https://www.freddiemac.com/pmms/docs/historicalweeklydata.xlsx"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
response = requests.get(url, headers=headers)
response.raise_for_status()
mortgage_rates = pd.read_excel(BytesIO(response.content))

In [5]:
mortgage_rates_df_clean = mortgage_rates.copy(deep=True)
mortgage_rates_df_clean = mortgage_rates_df_clean[mortgage_rates_df_clean['Unnamed: 0'].notnull()]

# take first row as header
mortgage_rates_df_clean.columns = mortgage_rates_df_clean.iloc[0]
mortgage_rates_df_clean = mortgage_rates_df_clean[1:].reset_index(drop=True)

week_parsed = pd.to_datetime(mortgage_rates_df_clean["Week"], errors="coerce")
mortgage_rates_df_clean = mortgage_rates_df_clean[week_parsed.notna()].copy()
mortgage_rates_df_clean["Week"] = week_parsed[week_parsed.notna()].dt.date

# renaming columns from base spreedsheet
mortgage_rates_df_clean.columns.values[0] = "Week"
mortgage_rates_df_clean.columns.values[1] = "U.S. 30 year FRM"
mortgage_rates_df_clean.columns.values[2] = "30 year fees & points"

mortgage_rates_df_clean.columns.values[3] = "U.S. 15 year FRM"
mortgage_rates_df_clean.columns.values[4] = "15 year fees & points"

mortgage_rates_df_clean.columns.values[5] = "U.S. 5/1 ARM"
mortgage_rates_df_clean.columns.values[6] = "5/1 year fees & points"

mortgage_rates_df_clean.columns.values[7] = "U.S. 5/1 ARM margin"
mortgage_rates_df_clean.columns.values[8] = "30 year FRM / 5/1 ARM spread"

#Truncate
mortgage_rates_df_clean = mortgage_rates_df_clean[mortgage_rates_df_clean['Week']>=pd.to_datetime("2016-01-01").date()]

mortgage_rates_df_clean['Week'] = pd.to_datetime(mortgage_rates_df_clean['Week'])
mortgage_rates_df_clean['Month'] = mortgage_rates_df_clean['Week'].dt.to_period('M').dt.to_timestamp()
mortgage_rates_df_clean.drop(columns=['Week'], inplace=True)
mortgage_rates_df_clean = mortgage_rates_df_clean.groupby('Month').mean().reset_index()

In [9]:
realtor_merge = pd.merge(realtor, realtor_hotness, how="outer", on=["month_date_yyyymm", "county_fips"])
realtor_merge[['city', 'state']] = realtor_merge['county_name_x'].str.split(', ', expand=True)
realtor_merge['month_date_yyyymm'] = pd.to_datetime(realtor_merge['month_date_yyyymm'], format='%Y%m')

In [ ]:
processed_data_pre_model = pd.merge(realtor_merge, mortgage_rates_df_clean, left_on="month_date_yyyymm", right_on="Month", how="left")

In [15]:
processed_data_pre_model.drop(columns=(['date','year','Month_x','Month_y']), inplace=True)
processed_data_pre_model.rename(columns={"month_date_yyyymm": "date"}, inplace=True)

In [16]:
processed_data_pre_model.to_csv("../data/processed/processed_data_pre_model.csv", index=False)